In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np

In [3]:
import sqlalchemy as sa
from sqlalchemy.orm import Session
from crmprtd import setup_logging
from pycds import Station, History

save_path = './comparison_forms/'

db_url = "postgresql://tongli1997@db.pcic.uvic.ca:5433/crmp?keepalives=1&keepalives_idle=300&keepalives_interval=300&keepalives_count=9&passfile=/workspaces/crmprtd/.pgpass"
log_file_path = save_path

engine = sa.create_engine(db_url, echo=False)
session = Session(engine)

session

In [4]:
path = '/workspaces/crmprtd/update_round2/input_tables/'
df = pd.read_excel(path + '20251230 PCDS Changes - AGRIWeather Round 3.xlsx', header = 0)   # pandas automatically uses openpyxl
id_issues = [
    "ID",
    "ID, Name",
]

name_issues = [
    "Name",
    "ID, Name",
]

df_id = df[
    df["ISSUE"].str.strip().isin(id_issues)
]

df_name = df[
    df["ISSUE"].str.strip().isin(name_issues)
]


df_id
# df_name

,ISSUE,network_id,native_id,station_name,min_obs_time,max_obs_time,lat,lon,elev
0,ID,10,AGAL001->ALCALILK,Alkali Lake,NaT,NaT,51.790278,-121.251111,677.000
1,ID,10,AGBCPRDNCK->BCPRDNCK,BCGPA Dawson Creek,NaT,NaT,55.772100,-120.143000,673.000
2,ID,10,AGBCPRBFLT->BCPRBFLT,Bearflats,NaT,NaT,56.246100,-121.308000,504.000
3,ID,10,AGOKNGBVST->OKNGBVST,Bella Vista,NaT,NaT,50.255100,-119.341300,437.000
4,ID,10,AGOKNGBNVN->OKNGBNVN,Benvoulin,NaT,NaT,49.872000,-119.449100,357.000
...,...,...,...,...,...,...,...,...,...
74,"ID, Name",10,AGCRESRIVW->CRESRIVW,North Lister (Riverview)->North Lister/Riverview,NaT,NaT,49.061800,-116.508100,609.000
75,"ID, Name",10,si107->SLVRHILL,Silver Hills Ranch->Silver Hills,2008-04-17,2010-09-14,50.253333,-118.690556,495.000
76,"ID, Name",10,su107->LOWNICOL,Sunshine Valley->Lower Nicola (Sunshine Valley),2005-08-05,2010-09-14,50.145833,-120.933333,560.832
103,ID,10,AGDLTACRES->DLTACRES,Crescent Island,NaT,NaT,49.125500,-123.034800,3.000


In [5]:
df_id

df_new = df_id[['native_id']].copy()

# Split on '->'
split_ids = df_new['native_id'].astype(str).str.split('->', expand=True)
df_new['old_native_id'] = split_ids[0].str.strip()
df_new['new_native_id'] = split_ids[1].str.strip()

df_new = df_new.drop(columns=['native_id'])

df_new

,old_native_id,new_native_id
0,AGAL001,ALCALILK
1,AGBCPRDNCK,BCPRDNCK
2,AGBCPRBFLT,BCPRBFLT
3,AGOKNGBVST,OKNGBVST
4,AGOKNGBNVN,OKNGBNVN
...,...,...
74,AGCRESRIVW,CRESRIVW
75,si107,SLVRHILL
76,su107,LOWNICOL
103,AGDLTACRES,DLTACRES


### In the db

In [6]:
import sqlalchemy as sa
from sqlalchemy.orm import Session
from crmprtd import setup_logging
from pycds import Station, History

db_url = "postgresql://tongli1997@db.pcic.uvic.ca:5433/crmp?keepalives=1&keepalives_idle=300&keepalives_interval=300&keepalives_count=9&passfile=/workspaces/crmprtd/.pgpass"
log_file_path = save_path

engine = sa.create_engine(db_url, echo=False)
session = Session(engine)

session

### Update the ID

In [7]:

check_sql = sa.text("""
SELECT station_id, native_id, network_id
FROM meta_station
WHERE native_id = :native_id
""")

with engine.connect() as conn:
    for _, row in df_new.iterrows():
        res = conn.execute(
            check_sql,
            {"native_id": row["old_native_id"]}
        ).fetchone()

        if res is None:
            print(f"❌ Station {row['old_native_id']} not found")
            continue

        db_native_id = res.native_id
        db_network_id = res.network_id

        if db_native_id == row["old_native_id"]:
            print(
                f"✅ OK to update station {res.station_id}: "
                f"{db_native_id} → {row['old_native_id']}, "
                f"It belongs to {db_network_id}"
            )
        else:
            print(
                f"⚠️ Mismatch for station {res.station_id}: "
                f"DB={db_native_id}, expected={row['old_native_id']}, "
                f"It belongs to {db_network_id}"

            )

✅ OK to update station 14798: AGAL001 → AGAL001, It belongs to 10
✅ OK to update station 14799: AGBCPRDNCK → AGBCPRDNCK, It belongs to 10
✅ OK to update station 14800: AGBCPRBFLT → AGBCPRBFLT, It belongs to 10
✅ OK to update station 14801: AGOKNGBVST → AGOKNGBVST, It belongs to 10
✅ OK to update station 14896: AGOKNGBNVN → AGOKNGBNVN, It belongs to 10
✅ OK to update station 1510: bo107 → bo107, It belongs to 10
✅ OK to update station 14898: AGBB003 → AGBB003, It belongs to 10
✅ OK to update station 14802: AGBCPRBUIK → AGBCPRBUIK, It belongs to 10
✅ OK to update station 14803: AGCRESCNYN → AGCRESCNYN, It belongs to 10
✅ OK to update station 14899: AGOKNGCWSTB → AGOKNGCWSTB, It belongs to 10
✅ OK to update station 1511: ch107 → ch107, It belongs to 10
✅ OK to update station 1512: co107 → co107, It belongs to 10
✅ OK to update station 1513: de107 → de107, It belongs to 10
✅ OK to update station 14901: AGBCPRDORV → AGBCPRDORV, It belongs to 10
✅ OK to update station 1515: do107 → do107, It

In [8]:
update_station_sql = sa.text("""
UPDATE meta_station
SET native_id = :new_native_id
WHERE native_id = :old_native_id
""")

with engine.begin() as conn:
    for _, row in df_new.iterrows():
        result = conn.execute(
            update_station_sql,
            {
                "old_native_id": row["old_native_id"],
                "new_native_id": row["new_native_id"],
            }
        )

        if result.rowcount == 1:
            print(
                f"✅ Updated station {row['old_native_id']}: "
                f"{row['old_native_id']} → {row['new_native_id']}"
            )
        else:
            # check why it failed
            res = conn.execute(
                check_sql,
                {"native_id": row["old_native_id"]}
            ).fetchone()

            if res is None:
                print(f"❌ Station {row['old_native_id']} not found")
            else:
                print(
                    f"⚠️ Skipped station {row['old_native_id']}: "
                    f"DB={res.native_id}, expected={row['old_native_id']}"
                )

✅ Updated station AGAL001: AGAL001 → ALCALILK
✅ Updated station AGBCPRDNCK: AGBCPRDNCK → BCPRDNCK
✅ Updated station AGBCPRBFLT: AGBCPRBFLT → BCPRBFLT
✅ Updated station AGOKNGBVST: AGOKNGBVST → OKNGBVST
✅ Updated station AGOKNGBNVN: AGOKNGBNVN → OKNGBNVN
✅ Updated station bo107: bo107 → BP002
✅ Updated station AGBB003: AGBB003 → BB003
✅ Updated station AGBCPRBUIK: AGBCPRBUIK → BCPRBUIK
✅ Updated station AGCRESCNYN: AGCRESCNYN → CRESCNYN
✅ Updated station AGOKNGCWSTB: AGOKNGCWSTB → OKNGCWSTB
✅ Updated station ch107: ch107 → CHASESTN
✅ Updated station co107: co107 → COLDWATR
✅ Updated station de107: de107 → DEEPCREE
✅ Updated station AGBCPRDORV: AGBCPRDORV → BCPRDORV
✅ Updated station do107: do107 → DOUGLAKE
✅ Updated station AGOKNGEKLN: AGOKNGEKLN → OKNGEKLN
✅ Updated station AGOKNGELSN: AGOKNGELSN → OKNGELSN
✅ Updated station AGBCPRFMTN: AGBCPRFMTN → BCPRFMTN
✅ Updated station AGBCPRFLHT: AGBCPRFLHT → BCPRFLHT
✅ Updated station AGBCPRFRCK: AGBCPRFRCK → BCPRFRCK
✅ Updated station AGOKNGG

In [9]:
verify_sql = sa.text("""
SELECT native_id
FROM meta_station
WHERE native_id = :native_id
""")

with engine.begin() as conn:
    for _, row in df_new.iterrows():
        # verify
        res = conn.execute(
            verify_sql,
            {"native_id": row["new_native_id"]}
        ).fetchone()

        if res and res.native_id == row["new_native_id"]:
            print(
                f"✅ Verified: station {row['new_native_id']} "
                f"native_id = {res.native_id}"
            )
        else:
            print(
                f"⚠️ Verification failed for station {row['new_native_id']}: "
                f"DB={res.native_id if res else None}, "
                f"expected={row['new_native_id']}"
            )

✅ Verified: station ALCALILK native_id = ALCALILK
✅ Verified: station BCPRDNCK native_id = BCPRDNCK
✅ Verified: station BCPRBFLT native_id = BCPRBFLT
✅ Verified: station OKNGBVST native_id = OKNGBVST
✅ Verified: station OKNGBNVN native_id = OKNGBNVN
✅ Verified: station BP002 native_id = BP002
✅ Verified: station BB003 native_id = BB003
✅ Verified: station BCPRBUIK native_id = BCPRBUIK
✅ Verified: station CRESCNYN native_id = CRESCNYN
✅ Verified: station OKNGCWSTB native_id = OKNGCWSTB
✅ Verified: station CHASESTN native_id = CHASESTN
✅ Verified: station COLDWATR native_id = COLDWATR
✅ Verified: station DEEPCREE native_id = DEEPCREE
✅ Verified: station BCPRDORV native_id = BCPRDORV
✅ Verified: station DOUGLAKE native_id = DOUGLAKE
✅ Verified: station OKNGEKLN native_id = OKNGEKLN
✅ Verified: station OKNGELSN native_id = OKNGELSN
✅ Verified: station BCPRFMTN native_id = BCPRFMTN
✅ Verified: station BCPRFLHT native_id = BCPRFLHT
✅ Verified: station BCPRFRCK native_id = BCPRFRCK
✅ Verified

## Update the name

In [17]:
df_name

,ISSUE,network_id,native_id,station_name,min_obs_time,max_obs_time,lat,lon,elev
72,"ID, Name",10,AGOKNGBLGO->OKGBLGO,Belgo 2->Belgo,NaT,NaT,49.876700,-119.384000,468.000
73,"ID, Name",10,di107->DIAMONDS,Diamond S->Diamond South,2007-09-14,2010-09-14,50.851930,-121.867490,450.000
74,"ID, Name",10,AGCRESRIVW->CRESRIVW,North Lister (Riverview)->North Lister/Riverview,NaT,NaT,49.061800,-116.508100,609.000
75,"ID, Name",10,si107->SLVRHILL,Silver Hills Ranch->Silver Hills,2008-04-17,2010-09-14,50.253333,-118.690556,495.000
76,"ID, Name",10,su107->LOWNICOL,Sunshine Valley->Lower Nicola (Sunshine Valley),2005-08-05,2010-09-14,50.145833,-120.933333,560.832
77,Name,10,SBC8,Belgo->Belgo (Old),2004-01-02,2010-09-14,49.879864,-119.373225,475.500
78,Name,10,sa89,Silver Creek ->Silver Creek (Old),2004-05-26,2010-09-14,50.632354,-119.385515,374.300


In [18]:
df_name

df_name_new =  df_name[['station_name', 'native_id']].reset_index(drop=True)


def split_station_name(name):
    if pd.isna(name):
        return pd.Series([None, None, 0])

    parts = [p.strip() for p in name.split("->")]

    if len(parts) == 2:
        old_name, new_name = parts
        n_names = 2
    else:
        old_name = new_name = parts[0]
        n_names = 1

    return pd.Series([old_name, new_name, n_names])
    
df_name = df_name_new.copy()

df_name[['old_name', 'new_name', 'n_names']] = (
    df_name['station_name'].apply(split_station_name)
)

split_ids = df_name["native_id"].astype(str).str.split("->", expand=True)

df_name["new_native_id"] = (
    split_ids[1]
    .str.strip()
    .fillna(df_name["native_id"].astype(str).str.strip())
)

df_name

,station_name,native_id,old_name,new_name,n_names,new_native_id
0,Belgo 2->Belgo,AGOKNGBLGO->OKGBLGO,Belgo 2,Belgo,2,OKGBLGO
1,Diamond S->Diamond South,di107->DIAMONDS,Diamond S,Diamond South,2,DIAMONDS
2,North Lister (Riverview)->North Lister/Riverview,AGCRESRIVW->CRESRIVW,North Lister (Riverview),North Lister/Riverview,2,CRESRIVW
3,Silver Hills Ranch->Silver Hills,si107->SLVRHILL,Silver Hills Ranch,Silver Hills,2,SLVRHILL
4,Sunshine Valley->Lower Nicola (Sunshine Valley),su107->LOWNICOL,Sunshine Valley,Lower Nicola (Sunshine Valley),2,LOWNICOL
5,Belgo->Belgo (Old),SBC8,Belgo,Belgo (Old),2,SBC8
6,Silver Creek ->Silver Creek (Old),sa89,Silver Creek,Silver Creek (Old),2,sa89


In [19]:
check_sql = sa.text("""
SELECT
    s.native_id,
    m.station_name
FROM meta_history AS m
JOIN meta_station AS s ON m.station_id = s.station_id
WHERE s.native_id = :native_id
""")

with engine.begin() as conn:
    for _, row in df_name.iterrows():

        native_id = row["new_native_id"]

        old_name  = row["old_name"]
        new_name  = row["new_name"]

        # --- 1. Read current DB values ---
        db_row = conn.execute(
            check_sql,
            {"native_id": native_id}
        ).fetchone()

        if db_row is None:
            print(f"❌ Station {native_id} not found in DB")
            continue

        # --- 2. Compare with expected OLD values ---

        if db_row.station_name != old_name:
            print(
                f"⚠️ Station {native_id} (DB, XLS values differ)\n"
                f"   DB : station name is {db_row.station_name}\n"
                f"   XLS: station name is {old_name}"
            )
        else:
            print(
                f"✅ Station {native_id}, {db_row.station_name} the names match "
            )


✅ Station OKGBLGO, Belgo 2 the names match 
✅ Station DIAMONDS, Diamond S the names match 
✅ Station CRESRIVW, North Lister (Riverview) the names match 
✅ Station SLVRHILL, Silver Hills Ranch the names match 
✅ Station LOWNICOL, Sunshine Valley the names match 
✅ Station SBC8, Belgo the names match 
✅ Station sa89, Silver Creek the names match 


In [20]:
df_name

,station_name,native_id,old_name,new_name,n_names,new_native_id
0,Belgo 2->Belgo,AGOKNGBLGO->OKGBLGO,Belgo 2,Belgo,2,OKGBLGO
1,Diamond S->Diamond South,di107->DIAMONDS,Diamond S,Diamond South,2,DIAMONDS
2,North Lister (Riverview)->North Lister/Riverview,AGCRESRIVW->CRESRIVW,North Lister (Riverview),North Lister/Riverview,2,CRESRIVW
3,Silver Hills Ranch->Silver Hills,si107->SLVRHILL,Silver Hills Ranch,Silver Hills,2,SLVRHILL
4,Sunshine Valley->Lower Nicola (Sunshine Valley),su107->LOWNICOL,Sunshine Valley,Lower Nicola (Sunshine Valley),2,LOWNICOL
5,Belgo->Belgo (Old),SBC8,Belgo,Belgo (Old),2,SBC8
6,Silver Creek ->Silver Creek (Old),sa89,Silver Creek,Silver Creek (Old),2,sa89


In [21]:
update_sql = sa.text("""
UPDATE meta_history AS m
SET station_name = :new_name
FROM meta_station AS s
WHERE m.station_id = s.station_id
  AND s.native_id  = :native_id
""")

with engine.begin() as conn:
    for _, row in df_name.iterrows():

        native_id = row["new_native_id"]

        old_name  = row["old_name"]
        new_name  = row["new_name"]

        # --- 1. Read current DB values ---
        db_row = conn.execute(
            check_sql,
            {"native_id": native_id}
        ).fetchone()

        if db_row is None:
            print(f"❌ Station {native_id} not found in DB")
            continue

        # --- 3. Perform update ---
        result = conn.execute(
            update_sql,
            {
                "native_id": native_id,
                "new_name": new_name,
            }
        )

        if result.rowcount == 1:
            print(
                f"✅ Updated station {native_id}: "
                f"({old_name}) → "
                f"({new_name})"
            )
        else:
            print(f"⚠️ Station {native_id}: no update applied")

✅ Updated station OKGBLGO: (Belgo 2) → (Belgo)
✅ Updated station DIAMONDS: (Diamond S) → (Diamond South)
✅ Updated station CRESRIVW: (North Lister (Riverview)) → (North Lister/Riverview)
✅ Updated station SLVRHILL: (Silver Hills Ranch) → (Silver Hills)
✅ Updated station LOWNICOL: (Sunshine Valley) → (Lower Nicola (Sunshine Valley))
✅ Updated station SBC8: (Belgo) → (Belgo (Old))
✅ Updated station sa89: (Silver Creek) → (Silver Creek (Old))


In [22]:
import numpy as np

with engine.begin() as conn:
    for _, row in df_name.iterrows():

        native_id = row["new_native_id"]

        old_name  = row["old_name"]
        new_name  = row["new_name"]

        # --- 1. Read current DB values ---
        db_row = conn.execute(
            check_sql,
            {"native_id": native_id}
        ).fetchone()

        if db_row is None:
            print(f"❌ Station {native_id} not found in DB")
            continue

        # --- 2. Compare with expected OLD values ---

        if db_row.station_name != new_name:
            print(
                f"⚠️ Station {native_id} (DB, XLS values differ)\n"
                f"   DB : station name is {db_row.station_name}\n"
                f"   XLS: station name is {new_name}"
            )
        else:
            print(
                f"✅ Station {native_id}, {db_row.station_name} the names match "
            )


✅ Station OKGBLGO, Belgo the names match 
✅ Station DIAMONDS, Diamond South the names match 
✅ Station CRESRIVW, North Lister/Riverview the names match 
✅ Station SLVRHILL, Silver Hills the names match 
✅ Station LOWNICOL, Lower Nicola (Sunshine Valley) the names match 
✅ Station SBC8, Belgo (Old) the names match 
✅ Station sa89, Silver Creek (Old) the names match 


### New data insert

In [23]:
new_issues = [
    "New"
]

df_new = df[
    df["ISSUE"].str.strip().isin(new_issues)
]

df_new

,ISSUE,network_id,native_id,station_name,min_obs_time,max_obs_time,lat,lon,elev
79,New,10,BCPRBSNC,Bison Creek,NaT,NaT,56.1737,-120.3743,655.0
80,New,10,CHILEAST,Chilliwack East,NaT,NaT,49.1865,-121.8564,11.0
81,New,10,BCPRCLAY,Clayhurst,NaT,NaT,56.1890,-120.0667,641.0
82,New,10,BCVHCTNT,Cottontail,NaT,NaT,54.0384,-123.7983,751.0
83,New,10,BCVHEDCK,Eden Creek,NaT,NaT,54.1079,-124.1615,724.0
84,New,10,BCVHFLNS,Fraser Lake North Shore,NaT,NaT,54.0801,-124.8464,677.0
85,New,10,OKNGGHDS,Giants Head South,NaT,NaT,49.5746,-119.6576,510.0
86,New,10,GFNURSRY,Grand Forks Nursery,NaT,NaT,49.0071,-118.4820,536.0
87,New,10,OKNGLMBY,Lumby,NaT,NaT,50.2237,-119.0261,523.0
88,New,10,NUKKOLAK,Nukko Lake,NaT,NaT,54.0891,-122.9754,772.0


In [24]:
from sqlalchemy import func

stations_created = []

# for _, row in df_new.iloc[0:2].iterrows():
for _, row in df_new.iterrows():
    
    name = row['station_name']
    nid  = row['native_id']

    # 1. Create Station
    st = Station(
        native_id=nid,
        publish=True,
        network_id=row['network_id'])


    session.add(st)
    session.flush()  # 得到 st.id

    h = History(
        station_id=st.id,
        station_name=name,
        lat=row['lat'],
        lon=row['lon'],
        elevation=row['elev'],
        province="BC",
        country="CA",
        the_geom=func.ST_SetSRID(func.ST_MakePoint(row['lon'], row['lat']), 4326))

    session.add(h)

    stations_created.append((name, st.id))

session.commit()

print("Created:", stations_created)

Created: [('Bison Creek', 14899), ('Chilliwack East', 14900), ('Clayhurst', 14901), ('Cottontail', 14902), ('Eden Creek', 14903), ('Fraser Lake North Shore', 14904), ('Giants Head South', 14905), ('Grand Forks Nursery', 14906), ('Lumby', 14907), ('Nukko Lake', 14908), ('Sumas Prairie', 14909), ('Oliver (Panorama)', 14910), ('Summerland Upper South', 14911), ('Trinity Valley', 14912), ('Tappen', 14913), ('Vanderhoof Northside', 14914), ('Vernon BX', 14915), ('West Kelowna', 14916), ('West Kelowna Nursery', 14917)]
